In [2]:
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

base_dir = r'C:\Users\Admin\Downloads\Compressed\FaceForensics++\data_flat'

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
)

train_generator = datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

validation_generator = datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)


Found 8453 images belonging to 2 classes.
Found 2112 images belonging to 2 classes.


In [4]:
from tensorflow.keras.applications import ResNet50

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])

model.fit(train_generator, validation_data=validation_generator, epochs=15)

model.save('resnet50.h5')


94765736/94765736 [==============================] - 28s 0us/step
Epoch 1/15
265/265 [==============================] - 723s 3s/step - loss: 0.7146 - accuracy: 0.5184 - val_loss: 0.6951 - val_accuracy: 0.5421
Epoch 2/15
265/265 [==============================] - 737s 3s/step - loss: 0.6895 - accuracy: 0.5385 - val_loss: 0.6948 - val_accuracy: 0.5213
Epoch 3/15
265/265 [==============================] - 707s 3s/step - loss: 0.6881 - accuracy: 0.5352 - val_loss: 0.6939 - val_accuracy: 0.5417
Epoch 4/15
265/265 [==============================] - 712s 3s/step - loss: 0.6883 - accuracy: 0.5366 - val_loss: 0.6957 - val_accuracy: 0.5213
Epoch 5/15
265/265 [==============================] - 735s 3s/step - loss: 0.6885 - accuracy: 0.5412 - val_loss: 0.6961 - val_accuracy: 0.5412
Epoch 6/15
265/265 [==============================] - 742s 3s/step - loss: 0.6864 - accuracy: 0.5514 - val_loss: 0.6957 - val_accuracy: 0.5426
Epoch 7/15
265/265 [==============================] - 722s 3s/step - loss: 0

C:\Users\Admin\anaconda3\envs\deepfake_detection\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [6]:
# Evaluate the model on the validation data
val_loss, val_accuracy = model.evaluate(validation_generator)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

from sklearn.metrics import confusion_matrix, classification_report

# Reset the validation generator to ensure predictions start from the beginning
validation_generator.reset()

# Get predictions on the validation set
preds = model.predict(validation_generator)
predicted_classes = (preds > 0.5).astype(int).ravel()

# Get true class labels
true_classes = validation_generator.classes
class_labels = list(validation_generator.class_indices.keys())

# Generate and print the confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)
print("Confusion Matrix:")
print(cm)

# Generate and print the classification report
report = classification_report(true_classes, predicted_classes, target_names=class_labels)
print("Classification Report:")
print(report)



66/66 [==============================] - 97s 1s/step - loss: 0.6977 - accuracy: 0.5346
Validation Loss: 0.6977
Validation Accuracy: 0.5346
66/66 [==============================] - 76s 1s/step
Confusion Matrix:
[[ 112  854]
 [ 111 1035]]
Classification Report:
              precision    recall  f1-score   support

        fake       0.50      0.12      0.19       966
        real       0.55      0.90      0.68      1146

    accuracy                           0.54      2112
   macro avg       0.53      0.51      0.44      2112
weighted avg       0.53      0.54      0.46      2112

